# Engine Training Notebook

Automated pipeline:

1. Read PGNs from `data/lichess_games_export.pgn`
2. Replay engine-side moves against local UCI analysis
3. Build structured move-level diagnostics
4. Train a tabular ML model to predict move quality loss
5. Convert model signals into concrete `Chess/Engine.pm` constant updates
6. Optionally patch `Chess/Engine.pm` directly (no LLM dependency)

## Prerequisites

- `scripts/fetch_own_lichess_pgns.sh` has been run.
- Python packages installed from `requirements.txt`.
- Local UCI engine works via `perl play.pl --uci`.

This notebook does not call external APIs.

In [ ]:
from __future__ import annotations

import difflib
import io
import json
import re
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chess
import chess.pgn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
REPO_ROOT = Path.cwd().resolve()  # Repository root used to resolve all relative paths.
PGN_PATH = REPO_ROOT / "data" / "lichess_games_export.pgn"  # Primary PGN input/output path for this notebook.
ENGINE_PM_PATH = REPO_ROOT / "Chess" / "Engine.pm"  # Target engine file for optional auto-patching.
MIGRATION_ROOT = REPO_ROOT / "engineMigration"  # Root directory for migration bundles.
MIGRATION_SUFFIX = "engine_training_recommendations"  # Flyway-style migration description suffix.
MIGRATION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")  # UTC migration bundle timestamp.
MIGRATION_DIR = MIGRATION_ROOT / f"V{MIGRATION_TIMESTAMP}__{MIGRATION_SUFFIX}"  # Bundle directory.
PATCH_PREVIEW_PATH = MIGRATION_DIR / "001_engine_patch.diff"  # Ordered migration patch artifact.
MODEL_REPORT_PATH = MIGRATION_DIR / "002_model_report.json"  # Ordered migration model artifact.
REPORT_PATH = MIGRATION_DIR / "003_training_report.json"  # Ordered migration run-report artifact.
MIGRATION_CHECK_CMD = f"scripts/apply_engine_migration.sh check {MIGRATION_DIR.name}"  # Verify patch apply state.
MIGRATION_APPLY_CMD = f"scripts/apply_engine_migration.sh apply {MIGRATION_DIR.name}"  # Apply patch via migration helper.

# Input source mode:
# - "existing_pgn": use PGN_PATH directly
# - "game_url_log": consume data/lichess_game_urls.log via fetch script into PGN_PATH
# - "lichess_db_urls": download one or more Lichess DB URLs and merge into PGN_PATH
INPUT_SOURCE_MODE = "existing_pgn"
GAME_URL_LOG_PATH = REPO_ROOT / "data" / "lichess_game_urls.log"  # URL log consumed by fetch script.
FETCH_SCRIPT_PATH = REPO_ROOT / "scripts" / "fetch_own_lichess_pgns.sh"  # Script that fetches per-game PGNs from URL log.
CLEAR_GAME_URL_LOG_AFTER_CONSUME = True  # If True, truncates URL log after successful fetch.

LICHESS_DB_URLS: List[str] = []  # Used only when INPUT_SOURCE_MODE="lichess_db_urls".
LICHESS_DB_TMP_DIR = REPO_ROOT / "data" / "tmp"  # Download cache for DB archives.
KEEP_DB_DOWNLOADS = False  # Keep downloaded DB files after PGN merge when True.

ENGINE_CMD = ["perl", "play.pl", "--uci", "--depth", "8"]  # UCI command used for position analysis.
ENGINE_NAME = "Perl-GigaChess"  # Name that identifies your bot inside PGN headers.
MAX_GAMES = 40  # Maximum number of engine games to consume from PGN file.
OPENING_PLY_LIMIT = 14  # Plies treated as opening for book-related diagnostics.
SEARCH_MOVETIME_MS = 250  # Time per position to query the engine's preferred move.
EVAL_MOVETIME_MS = 250  # Time for post-played-move evaluation queries.
ANALYZE_PLAYED_EVAL = True  # If True, evaluates the played move to compute cp_loss.
ENABLE_BOOK_COMPARE = True  # If True, compares OwnBook=true vs OwnBook=false in opening.

MIN_ROWS_FOR_MODEL = 24  # Minimum labeled rows required before fitting ML model.
SEVERE_CP_LOSS = 150  # CP loss threshold used to tag severe tactical errors.



RUN_POST_PATCH_EVAL = False  # If True, reruns full analysis after applying patch for true before/after metrics.
POST_PATCH_MAX_GAMES = MAX_GAMES  # Number of games to use in post-patch verification run.
APPLY_ENGINE_PATCH = False  # If True, writes proposed constant updates directly to Engine.pm.
WRITE_PATCH_PREVIEW = True  # If True, writes unified diff preview artifact for migration bundles.

PIECE_VALUE_CP = {
    chess.PAWN: 100,    # Pawn value for material proxy feature engineering.
    chess.KNIGHT: 320,  # Knight value for material proxy feature engineering.
    chess.BISHOP: 330,  # Bishop value for material proxy feature engineering.
    chess.ROOK: 500,    # Rook value for material proxy feature engineering.
    chess.QUEEN: 900,   # Queen value for material proxy feature engineering.
    chess.KING: 0,      # King excluded from material balance arithmetic.
}

print(f"Repo root: {REPO_ROOT}")
print(f"Source mode: {INPUT_SOURCE_MODE}")
print(f"PGN path: {PGN_PATH}")
print(f"Engine: {' '.join(ENGINE_CMD)}")
print(f"Migration bundle: {MIGRATION_DIR}")

In [ ]:
@dataclass
class UciResult:
    bestmove: Optional[str]
    score_cp: Optional[int]
    depth: Optional[int]
    raw_lines: List[str]


class UciEngineSession:
    def __init__(self, cmd: List[str], cwd: Path):
        self.proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            bufsize=1,
        )
        self._init_uci()

    def _send(self, command: str) -> None:
        assert self.proc.stdin is not None
        self.proc.stdin.write(command + "\n")
        self.proc.stdin.flush()

    def _read_until_prefix(self, prefix: str) -> List[str]:
        assert self.proc.stdout is not None
        lines: List[str] = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError("UCI process closed unexpectedly")
            line = line.rstrip("\n")
            lines.append(line)
            if line.startswith(prefix):
                return lines

    def _init_uci(self) -> None:
        self._send("uci")
        self._read_until_prefix("uciok")
        self._send("setoption name MoveOverhead value 0")
        self._send("isready")
        self._read_until_prefix("readyok")

    @staticmethod
    def _score_from_info(line: str) -> Optional[int]:
        cp_match = re.search(r"score cp (-?\d+)", line)
        if cp_match:
            return int(cp_match.group(1))

        mate_match = re.search(r"score mate (-?\d+)", line)
        if mate_match:
            mate = int(mate_match.group(1))
            return 900000 if mate > 0 else -900000

        return None

    def query(self, moves_uci: List[str], ownbook: bool, movetime_ms: int) -> UciResult:
        book_val = "true" if ownbook else "false"
        self._send(f"setoption name OwnBook value {book_val}")

        if moves_uci:
            self._send("position startpos moves " + " ".join(moves_uci))
        else:
            self._send("position startpos")

        self._send(f"go movetime {int(movetime_ms)}")
        lines = self._read_until_prefix("bestmove")

        bestmove = None
        score_cp = None
        depth = None
        for line in lines:
            if line.startswith("info depth"):
                d_match = re.search(r"info depth (\d+)", line)
                if d_match:
                    depth = int(d_match.group(1))
                parsed = self._score_from_info(line)
                if parsed is not None:
                    score_cp = parsed
            elif line.startswith("bestmove "):
                parts = line.split()
                bestmove = parts[1] if len(parts) > 1 else None

        return UciResult(bestmove=bestmove, score_cp=score_cp, depth=depth, raw_lines=lines)

    def close(self) -> None:
        if self.proc.poll() is not None:
            return
        try:
            self._send("quit")
        except Exception:
            pass
        self.proc.wait(timeout=3)

In [ ]:
def _nonempty_log_entries(path: Path) -> List[str]:
    if not path.exists():
        return []
    entries: List[str] = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        entries.append(line)
    return entries


def _truncate_file(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("", encoding="utf-8")


def _append_plain_pgn(src: Path, out_handle) -> None:
    with src.open("r", encoding="utf-8", errors="ignore") as handle:
        while True:
            chunk = handle.read(1 << 20)
            if not chunk:
                break
            out_handle.write(chunk)


def _append_zst_pgn(src: Path, out_handle) -> None:
    try:
        import zstandard as zstd
    except ImportError as exc:
        raise RuntimeError("zstandard package is required to decode .zst files") from exc

    with src.open("rb") as compressed:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed) as reader:
            text_reader = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
            while True:
                chunk = text_reader.read(1 << 20)
                if not chunk:
                    break
                out_handle.write(chunk)


def _download_lichess_db_urls(urls: List[str], tmp_dir: Path, merged_pgn: Path, keep_downloads: bool) -> Path:
    if not urls:
        raise ValueError("LICHESS_DB_URLS is empty")

    tmp_dir.mkdir(parents=True, exist_ok=True)
    merged_pgn.parent.mkdir(parents=True, exist_ok=True)

    with merged_pgn.open("w", encoding="utf-8") as out:
        for idx, url in enumerate(urls, start=1):
            base_name = Path(url.split("?")[0]).name or f"lichess_{idx}.pgn"
            dest = tmp_dir / base_name
            cmd = ["curl", "-fL", "--retry", "3", "--retry-delay", "1", url, "-o", str(dest)]
            print("Downloading:", url)
            result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(f"curl failed for {url}: {result.stderr.strip() or result.stdout.strip()}")

            if dest.suffix.lower() == ".zst":
                _append_zst_pgn(dest, out)
            else:
                _append_plain_pgn(dest, out)
            out.write("\n\n")

            if not keep_downloads:
                dest.unlink(missing_ok=True)

    return merged_pgn


def _consume_game_url_log(log_path: Path, out_pgn: Path) -> Path:
    entries = _nonempty_log_entries(log_path)
    if not entries:
        raise ValueError(f"No URLs/game IDs found in {log_path}")
    if not FETCH_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Missing fetch script: {FETCH_SCRIPT_PATH}")

    cmd = [str(FETCH_SCRIPT_PATH), str(log_path), str(out_pgn)]
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    if CLEAR_GAME_URL_LOG_AFTER_CONSUME:
        _truncate_file(log_path)
        print(f"Cleared consumed URL log: {log_path}")

    return out_pgn


def prepare_input_pgn() -> Path:
    mode = INPUT_SOURCE_MODE.strip().lower()
    if mode == "existing_pgn":
        if not PGN_PATH.exists():
            raise FileNotFoundError(f"Missing PGN file: {PGN_PATH}")
        return PGN_PATH

    if mode == "game_url_log":
        return _consume_game_url_log(GAME_URL_LOG_PATH, PGN_PATH)

    if mode == "lichess_db_urls":
        return _download_lichess_db_urls(LICHESS_DB_URLS, LICHESS_DB_TMP_DIR, PGN_PATH, KEEP_DB_DOWNLOADS)

    raise ValueError(f"Unsupported INPUT_SOURCE_MODE: {INPUT_SOURCE_MODE}")


ACTIVE_PGN_PATH = prepare_input_pgn()
print(f"Using PGN: {ACTIVE_PGN_PATH}")

In [ ]:
def material_eval_cp(board: chess.Board, side: chess.Color) -> int:
    ours = 0
    theirs = 0
    for square, piece in board.piece_map().items():
        value = PIECE_VALUE_CP.get(piece.piece_type, 0)
        if piece.color == side:
            ours += value
        else:
            theirs += value
    return ours - theirs


def king_ring_squares(board: chess.Board, color: chess.Color) -> List[int]:
    king_sq = board.king(color)
    if king_sq is None:
        return []
    ring: List[int] = []
    for sq in chess.SquareSet(chess.BB_KING_ATTACKS[king_sq]):
        ring.append(int(sq))
    return ring


def king_danger_proxy(board: chess.Board, side: chess.Color) -> Dict[str, int]:
    # Side = engine side to move in this position.
    king_sq = board.king(side)
    if king_sq is None:
        return {
            "king_in_check": 0,
            "king_ring_attacked": 0,
            "king_ring_undefended": 0,
        }

    enemy = not side
    ring = king_ring_squares(board, side)
    ring_attacked = 0
    ring_undefended = 0
    for sq in ring:
        if board.is_attacked_by(enemy, sq):
            ring_attacked += 1
            if not board.is_attacked_by(side, sq):
                ring_undefended += 1

    return {
        "king_in_check": int(board.is_check()),
        "king_ring_attacked": int(ring_attacked),
        "king_ring_undefended": int(ring_undefended),
    }


def load_engine_games(path: Path, engine_name: str, max_games: int = 0) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing PGN file: {path}")

    games: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        while True:
            game = chess.pgn.read_game(handle)
            if game is None:
                break

            white = (game.headers.get("White") or "").strip()
            black = (game.headers.get("Black") or "").strip()
            if white != engine_name and black != engine_name:
                continue

            engine_is_white = white == engine_name
            games.append(
                {
                    "headers": dict(game.headers),
                    "engine_is_white": engine_is_white,
                    "moves": list(game.mainline_moves()),
                }
            )

            if max_games and len(games) >= max_games:
                break

    return games


games = load_engine_games(ACTIVE_PGN_PATH, ENGINE_NAME, MAX_GAMES)
print(f"Loaded {len(games)} engine games from {ACTIVE_PGN_PATH}")

In [ ]:
def analyze_game(game: Dict[str, Any], engine: UciEngineSession) -> List[Dict[str, Any]]:
    headers = game["headers"]
    moves = game["moves"]
    engine_is_white = game["engine_is_white"]

    board = chess.Board()
    played_prefix: List[str] = []
    rows: List[Dict[str, Any]] = []

    for ply_idx, move in enumerate(moves, start=1):
        engine_to_move = (board.turn == chess.WHITE and engine_is_white) or (board.turn == chess.BLACK and not engine_is_white)

        if engine_to_move:
            engine_side = board.turn
            played_uci = move.uci()
            move_number = board.fullmove_number
            san = board.san(move)
            fen_before = board.fen()

            no_book = engine.query(played_prefix, ownbook=False, movetime_ms=SEARCH_MOVETIME_MS)
            with_book = None
            if ENABLE_BOOK_COMPARE and ply_idx <= OPENING_PLY_LIMIT:
                with_book = engine.query(played_prefix, ownbook=True, movetime_ms=max(50, SEARCH_MOVETIME_MS // 2))

            cp_best = no_book.score_cp
            cp_played = None
            cp_loss = None

            if ANALYZE_PLAYED_EVAL:
                after_played = engine.query(played_prefix + [played_uci], ownbook=False, movetime_ms=EVAL_MOVETIME_MS)
                if after_played.score_cp is not None:
                    cp_played = -after_played.score_cp

            if cp_best is not None and cp_played is not None:
                cp_loss = cp_best - cp_played

            board_after = board.copy(stack=False)
            board_after.push(move)
            opp_checks_after = sum(1 for m in board_after.legal_moves if board_after.gives_check(m))

            kd = king_danger_proxy(board, engine_side)

            rows.append(
                {
                    "site": headers.get("Site"),
                    "date": headers.get("Date"),
                    "result": headers.get("Result"),
                    "engine_color": "white" if engine_is_white else "black",
                    "ply": ply_idx,
                    "move_number": move_number,
                    "played_san": san,
                    "played_uci": played_uci,
                    "best_uci_nobook": no_book.bestmove,
                    "best_cp_nobook": cp_best,
                    "played_cp": cp_played,
                    "cp_loss": cp_loss,
                    "opening_phase": int(ply_idx <= OPENING_PLY_LIMIT),
                    "played_is_capture": int(board.is_capture(move)),
                    "played_gives_check": int(board.gives_check(move)),
                    "opponent_checks_after": int(opp_checks_after),
                    "book_uci": with_book.bestmove if with_book else None,
                    "book_agrees_played": int(bool(with_book and with_book.bestmove == played_uci)),
                    "book_differs_nobook": int(bool(with_book and with_book.bestmove and no_book.bestmove and with_book.bestmove != no_book.bestmove)),
                    "best_matches_played": int(no_book.bestmove == played_uci),
                    "fen_before": fen_before,
                    "legal_moves": int(board.legal_moves.count()),
                    "material_cp": int(material_eval_cp(board, engine_side)),
                    "king_in_check": int(kd["king_in_check"]),
                    "king_ring_attacked": int(kd["king_ring_attacked"]),
                    "king_ring_undefended": int(kd["king_ring_undefended"]),
                }
            )

        played_prefix.append(move.uci())
        board.push(move)

    return rows

In [ ]:
all_rows: List[Dict[str, Any]] = []
engine = UciEngineSession(ENGINE_CMD, REPO_ROOT)
try:
    for i, game in enumerate(games, start=1):
        rows = analyze_game(game, engine)
        all_rows.extend(rows)
        print(f"Analyzed game {i}/{len(games)} -> {len(rows)} engine moves")
finally:
    engine.close()

df = pd.DataFrame(all_rows)
print(f"Total engine moves analyzed: {len(df)}")
if len(df):
    display(df.head(10))

In [ ]:
def summarize_dataset(df: pd.DataFrame) -> Dict[str, Any]:
    if df.empty:
        return {"engine_moves": 0, "games": 0}

    cp_df = df[df["cp_loss"].notna()].copy()
    critical = cp_df[cp_df["cp_loss"] >= 80]
    severe = cp_df[cp_df["cp_loss"] >= SEVERE_CP_LOSS]

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "games": int(df["site"].nunique()),
        "engine_moves": int(len(df)),
        "moves_with_eval": int(len(cp_df)),
        "bestmove_match_rate": float(df["best_matches_played"].mean()) if len(df) else None,
        "avg_cp_loss": float(cp_df["cp_loss"].mean()) if len(cp_df) else None,
        "critical_80cp": int(len(critical)),
        "severe_150cp": int(len(severe)),
        "capture_critical": int(len(critical[critical["played_is_capture"] == 1])) if len(critical) else 0,
        "quiet_critical": int(len(critical[critical["played_is_capture"] == 0])) if len(critical) else 0,
        "critical_with_opp_checks": int(len(critical[critical["opponent_checks_after"] > 0])) if len(critical) else 0,
        "book_disagreements_opening": int(df[df["opening_phase"] == 1]["book_differs_nobook"].sum()) if len(df) else 0,
    }


summary = summarize_dataset(df)
print(json.dumps(summary, indent=2))

if not df.empty and df["cp_loss"].notna().any():
    top = df[df["cp_loss"].notna()].sort_values("cp_loss", ascending=False).head(20)
    display(top[["site", "move_number", "played_san", "played_uci", "best_uci_nobook", "best_cp_nobook", "played_cp", "cp_loss", "played_is_capture", "king_ring_attacked", "opponent_checks_after", "fen_before"]])

## Baseline Visualizations

These charts show **before-change** behavior from the current dataset.

In [ ]:
if df.empty:
    print("No rows to visualize.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    cp = df[df["cp_loss"].notna()]["cp_loss"]
    if len(cp):
        axes[0].hist(cp, bins=20, color="#4472c4", alpha=0.85)
        axes[0].set_title("CP Loss Distribution")
        axes[0].set_xlabel("cp_loss")
        axes[0].set_ylabel("count")
    else:
        axes[0].text(0.5, 0.5, "No cp_loss data", ha="center", va="center")
        axes[0].set_title("CP Loss Distribution")

    by_move = df[df["cp_loss"].notna()].groupby("move_number")["cp_loss"].mean().reset_index()
    if len(by_move):
        axes[1].plot(by_move["move_number"], by_move["cp_loss"], marker="o", color="#70ad47")
        axes[1].set_title("Avg CP Loss by Move Number")
        axes[1].set_xlabel("move_number")
        axes[1].set_ylabel("avg cp_loss")
    else:
        axes[1].text(0.5, 0.5, "No cp_loss data", ha="center", va="center")
        axes[1].set_title("Avg CP Loss by Move Number")

    cap = df[df["cp_loss"].notna()].groupby("played_is_capture")["cp_loss"].mean()
    labels = ["quiet", "capture"]
    vals = [float(cap.get(0, 0.0)), float(cap.get(1, 0.0))]
    axes[2].bar(labels, vals, color=["#ed7d31", "#a5a5a5"])
    axes[2].set_title("Avg CP Loss: Quiet vs Capture")
    axes[2].set_ylabel("avg cp_loss")

    plt.tight_layout()
    plt.show()

if not df.empty:
    risk_df = pd.DataFrame(
        {
            "metric": [
                "bestmove_match_rate",
                "avg_cp_loss",
                "critical_80cp",
                "severe_150cp",
                "book_disagreements_opening",
            ],
            "value": [
                summary.get("bestmove_match_rate"),
                summary.get("avg_cp_loss"),
                summary.get("critical_80cp"),
                summary.get("severe_150cp"),
                summary.get("book_disagreements_opening"),
            ],
        }
    )
    display(risk_df)

## Train ML Model (No LLM)

Model: `GradientBoostingRegressor` on move-level diagnostics to predict `cp_loss`.

Then we use model importances + aggregate risk statistics to propose constant updates.

In [ ]:
feature_cols = [
    "opening_phase",
    "played_is_capture",
    "played_gives_check",
    "opponent_checks_after",
    "book_differs_nobook",
    "best_matches_played",
    "legal_moves",
    "material_cp",
    "king_in_check",
    "king_ring_attacked",
    "king_ring_undefended",
]

model_metrics: Dict[str, Any] = {}
feature_importance: List[Tuple[str, float]] = []
trained_model = None

df_eval = df[df["cp_loss"].notna()].copy()
if len(df_eval) < MIN_ROWS_FOR_MODEL:
    print(f"Not enough labeled rows for robust ML ({len(df_eval)} < {MIN_ROWS_FOR_MODEL}).")
else:
    X = df_eval[feature_cols].fillna(0.0)
    y = df_eval["cp_loss"].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    model = GradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    importances = model.feature_importances_
    pairs = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

    trained_model = model
    model_metrics = {
        "rows": int(len(df_eval)),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "mae": float(mae),
        "r2": float(r2),
    }
    feature_importance = [(name, float(score)) for name, score in pairs]

    print("Model metrics:")
    print(json.dumps(model_metrics, indent=2))

    print("\nFeature importance:")
    for name, score in feature_importance:
        print(f"- {name}: {score:.4f}")

## Feature Importance Visualization

In [ ]:
if feature_importance:
    fi_df = pd.DataFrame(feature_importance, columns=["feature", "importance"]).sort_values("importance", ascending=True)
    plt.figure(figsize=(8, 5))
    plt.barh(fi_df["feature"], fi_df["importance"], color="#5b9bd5")
    plt.title("Model Feature Importances")
    plt.xlabel("importance")
    plt.tight_layout()
    plt.show()
else:
    print("No model importances available.")

In [ ]:
CONST_PATTERN = re.compile(r"^(\s*use constant\s+([A-Z0-9_]+)\s*=>\s*)([^;]+)(;\s*(?:#.*)?)$")


def parse_engine_constants(path: Path) -> Dict[str, float]:
    constants: Dict[str, float] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        m = CONST_PATTERN.match(line)
        if not m:
            continue
        key = m.group(2)
        raw = m.group(3).strip()
        try:
            constants[key] = float(raw)
        except ValueError:
            # Skip symbolic constants that are not plain numbers.
            continue
    return constants


def clamp(v: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, v))


def scaled(base: float, factor: float, lo: float, hi: float) -> float:
    return clamp(base * factor, lo, hi)


def propose_constant_updates(df_eval: pd.DataFrame, summary: Dict[str, Any], constants: Dict[str, float], feature_importance: List[Tuple[str, float]]) -> Dict[str, float]:
    if df_eval.empty:
        return {}

    importance_map = {k: v for k, v in feature_importance}

    capture_loss = df_eval[df_eval["played_is_capture"] == 1]["cp_loss"].mean()
    quiet_loss = df_eval[df_eval["played_is_capture"] == 0]["cp_loss"].mean()
    capture_delta = float((capture_loss - quiet_loss) if pd.notna(capture_loss) and pd.notna(quiet_loss) else 0.0)

    check_risk = float(
        df_eval[df_eval["opponent_checks_after"] > 0]["cp_loss"].mean() -
        df_eval[df_eval["opponent_checks_after"] == 0]["cp_loss"].mean()
    ) if (df_eval["opponent_checks_after"] > 0).any() and (df_eval["opponent_checks_after"] == 0).any() else 0.0

    severe_rate = float((df_eval["cp_loss"] >= SEVERE_CP_LOSS).mean())
    avg_cp_loss = float(df_eval["cp_loss"].mean())

    updates: Dict[str, float] = {}

    # Capture-risk tuning.
    if capture_delta > 12 or importance_map.get("played_is_capture", 0.0) >= 0.10:
        if "UNSAFE_CAPTURE_HANGING_BONUS" in constants:
            factor = 1.0 + min(0.45, max(0.10, capture_delta / 220.0))
            updates["UNSAFE_CAPTURE_HANGING_BONUS"] = round(scaled(constants["UNSAFE_CAPTURE_HANGING_BONUS"], factor, 8, 80))
        if "UNSAFE_CAPTURE_KING_EXPOSURE_WEIGHT" in constants:
            factor = 1.0 + min(0.50, max(0.08, capture_delta / 260.0))
            updates["UNSAFE_CAPTURE_KING_EXPOSURE_WEIGHT"] = round(scaled(constants["UNSAFE_CAPTURE_KING_EXPOSURE_WEIGHT"], factor, 1, 12))

    # King-safety tuning.
    king_signal = (
        importance_map.get("king_ring_attacked", 0.0)
        + importance_map.get("king_ring_undefended", 0.0)
        + importance_map.get("opponent_checks_after", 0.0)
    )
    if check_risk > 10 or king_signal >= 0.18:
        if "KING_DANGER_CHECK_PENALTY" in constants:
            factor = 1.0 + min(0.45, max(0.08, check_risk / 240.0))
            updates["KING_DANGER_CHECK_PENALTY"] = round(scaled(constants["KING_DANGER_CHECK_PENALTY"], factor, 4, 28))
        if "KING_DANGER_RING_ATTACK_PENALTY" in constants:
            factor = 1.0 + min(0.35, max(0.05, check_risk / 300.0))
            updates["KING_DANGER_RING_ATTACK_PENALTY"] = round(scaled(constants["KING_DANGER_RING_ATTACK_PENALTY"], factor, 1, 8))
        if "LMR_KING_DANGER_THRESHOLD" in constants:
            factor = 1.0 - min(0.25, max(0.05, check_risk / 500.0))
            updates["LMR_KING_DANGER_THRESHOLD"] = round(scaled(constants["LMR_KING_DANGER_THRESHOLD"], factor, 5, 24))

    # Tactical horizon tuning.
    if severe_rate >= 0.10 or avg_cp_loss >= 65:
        if "QUIESCE_CHECK_BONUS" in constants:
            factor = 1.0 + min(0.45, max(0.10, severe_rate))
            updates["QUIESCE_CHECK_BONUS"] = round(scaled(constants["QUIESCE_CHECK_BONUS"], factor, 40, 260))
        if "QUIESCE_CHECK_MAX_DEPTH" in constants:
            updates["QUIESCE_CHECK_MAX_DEPTH"] = round(clamp(constants["QUIESCE_CHECK_MAX_DEPTH"] + 1, 1, 3))

    # Stability/depth tuning if model sees broad error and low move agreement.
    match_rate = summary.get("bestmove_match_rate")
    if match_rate is not None and match_rate < 0.55:
        if "EXTRA_DEPTH_ON_UNSTABLE" in constants:
            updates["EXTRA_DEPTH_ON_UNSTABLE"] = round(clamp(constants["EXTRA_DEPTH_ON_UNSTABLE"] + 1, 1, 8))

    return updates


engine_constants = parse_engine_constants(ENGINE_PM_PATH)
proposed_updates = propose_constant_updates(df_eval, summary, engine_constants, feature_importance)
updates_df = pd.DataFrame(columns=["constant", "old", "new", "delta"])

if not proposed_updates:
    print("No automatic constant updates proposed from current sample.")
else:
    rows = []
    for key, new_value in sorted(proposed_updates.items()):
        old_value = engine_constants.get(key)
        rows.append(
            {
                "constant": key,
                "old": old_value,
                "new": new_value,
                "delta": (new_value - old_value) if old_value is not None else None,
            }
        )
    updates_df = pd.DataFrame(rows)
    display(updates_df)

## Constants Before/After

In [ ]:
if updates_df.empty:
    print("No proposed constant updates to visualize.")
else:
    viz_df = updates_df.copy().sort_values("delta")
    display(viz_df)

    y = np.arange(len(viz_df))
    h = 0.38
    plt.figure(figsize=(10, max(3, 0.6 * len(viz_df))))
    plt.barh(y - h/2, viz_df["old"], height=h, label="before", color="#a5a5a5")
    plt.barh(y + h/2, viz_df["new"], height=h, label="after (proposed)", color="#4472c4")
    plt.yticks(y, viz_df["constant"])
    plt.xlabel("constant value")
    plt.title("Engine Constant Proposals: Before vs After")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def format_value_like(old_raw: str, value: float) -> str:
    old_raw = old_raw.strip()
    if re.search(r"\.", old_raw):
        txt = f"{float(value):.4f}" if abs(float(value)) < 1 else f"{float(value):.3f}"
        txt = txt.rstrip("0").rstrip(".")
        return txt if txt else "0"
    return str(int(round(float(value))))


def apply_constant_updates(path: Path, updates: Dict[str, float]) -> Tuple[str, str, List[str]]:
    old_text = path.read_text(encoding="utf-8")
    changed_keys: List[str] = []

    new_lines: List[str] = []
    for line in old_text.splitlines(keepends=True):
        m = CONST_PATTERN.match(line.rstrip("\n"))
        if not m:
            new_lines.append(line)
            continue

        prefix, key, old_raw, suffix = m.groups()
        if key not in updates:
            new_lines.append(line)
            continue

        new_raw = format_value_like(old_raw, updates[key])
        rebuilt = f"{prefix}{new_raw}{suffix}\n"
        new_lines.append(rebuilt)
        changed_keys.append(key)

    new_text = "".join(new_lines)
    return old_text, new_text, sorted(set(changed_keys))


patch_applied = False
changed_constants: List[str] = []

model_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "summary": summary,
    "model_metrics": model_metrics,
    "feature_importance": [{"feature": k, "importance": v} for k, v in feature_importance],
    "proposed_constant_updates": proposed_updates,
}
MODEL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_REPORT_PATH.write_text(json.dumps(model_report, indent=2), encoding="utf-8")
print(f"Wrote model report: {MODEL_REPORT_PATH}")

if proposed_updates:
    old_text, new_text, changed = apply_constant_updates(ENGINE_PM_PATH, proposed_updates)
    diff = "".join(
        difflib.unified_diff(
            old_text.splitlines(keepends=True),
            new_text.splitlines(keepends=True),
            fromfile=str(ENGINE_PM_PATH),
            tofile=str(ENGINE_PM_PATH),
        )
    )

    if WRITE_PATCH_PREVIEW:
        PATCH_PREVIEW_PATH.write_text(diff, encoding="utf-8")
        print(f"Wrote patch preview: {PATCH_PREVIEW_PATH}")

    changed_constants = changed
    if APPLY_ENGINE_PATCH:
        ENGINE_PM_PATH.write_text(new_text, encoding="utf-8")
        patch_applied = True
        print(f"Applied updates to {ENGINE_PM_PATH}")
        print("Changed constants:", ", ".join(changed_constants))
    else:
        print("APPLY_ENGINE_PATCH is False. No file changes applied.")
        print("Proposed changes:", ", ".join(changed_constants))
        if diff:
            print("\nPatch preview diff:\n")
            print(diff)
        print("Apply this migration with:")
        print(f"  {MIGRATION_CHECK_CMD}")
        print(f"  {MIGRATION_APPLY_CMD}")
else:
    print("No patch preview produced because no updates were proposed.")

## Post-Patch Verification (Optional)

Enable `RUN_POST_PATCH_EVAL=True` and `APPLY_ENGINE_PATCH=True` to run a true after-change comparison.

In [ ]:
summary_after: Dict[str, Any] | None = None

if RUN_POST_PATCH_EVAL and patch_applied:
    verify_games = games[:POST_PATCH_MAX_GAMES] if POST_PATCH_MAX_GAMES else games
    rows_after: List[Dict[str, Any]] = []
    verify_engine = UciEngineSession(ENGINE_CMD, REPO_ROOT)
    try:
        for i, game in enumerate(verify_games, start=1):
            game_rows = analyze_game(game, verify_engine)
            rows_after.extend(game_rows)
            print(f"Post-patch game {i}/{len(verify_games)} -> {len(game_rows)} rows")
    finally:
        verify_engine.close()

    df_after = pd.DataFrame(rows_after)
    summary_after = summarize_dataset(df_after)
    print("Post-patch summary:")
    print(json.dumps(summary_after, indent=2))

    metric_keys = ["bestmove_match_rate", "avg_cp_loss", "critical_80cp", "severe_150cp"]
    comp_rows = []
    for k in metric_keys:
        comp_rows.append({"metric": k, "before": summary.get(k), "after": summary_after.get(k)})
    comp_df = pd.DataFrame(comp_rows)
    display(comp_df)

    # Plot side-by-side where numeric values are available.
    plot_df = comp_df.dropna().copy()
    if len(plot_df):
        x = np.arange(len(plot_df))
        w = 0.38
        plt.figure(figsize=(9, 4))
        plt.bar(x - w/2, plot_df["before"], width=w, label="before", color="#a5a5a5")
        plt.bar(x + w/2, plot_df["after"], width=w, label="after", color="#70ad47")
        plt.xticks(x, plot_df["metric"], rotation=20)
        plt.title("Before vs After Patch Metrics")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Post-patch verification skipped. Set RUN_POST_PATCH_EVAL=True and APPLY_ENGINE_PATCH=True.")

In [ ]:
# Always write a compact run report for automation.
full_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pgn_path": str(ACTIVE_PGN_PATH),
    "engine_cmd": ENGINE_CMD,
    "input_source_mode": INPUT_SOURCE_MODE,
    "migration": {
        "dir": str(MIGRATION_DIR),
        "name": MIGRATION_DIR.name,
        "check_cmd": MIGRATION_CHECK_CMD,
        "apply_cmd": MIGRATION_APPLY_CMD,
    },
    "config": {
        "engine_name": ENGINE_NAME,
        "max_games": MAX_GAMES,
        "opening_ply_limit": OPENING_PLY_LIMIT,
        "search_movetime_ms": SEARCH_MOVETIME_MS,
        "eval_movetime_ms": EVAL_MOVETIME_MS,
        "analyze_played_eval": ANALYZE_PLAYED_EVAL,
        "enable_book_compare": ENABLE_BOOK_COMPARE,
        "min_rows_for_model": MIN_ROWS_FOR_MODEL,
        "severe_cp_loss": SEVERE_CP_LOSS,
    },
    "summary_before": summary,
    "summary_after": summary_after,
    "model_metrics": model_metrics,
    "feature_importance": [{"feature": k, "importance": v} for k, v in feature_importance],
    "proposed_constant_updates": proposed_updates,
    "applied_patch": bool(patch_applied),
    "changed_constants": changed_constants,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(full_report, indent=2), encoding="utf-8")
print(f"Wrote run report: {REPORT_PATH}")